# Description

- 📦 **Setup & Data Loading**
  - Installs `langchain-openai` and `langchain-core`.
  - Loads the OpenAI API key from Colab `userdata`.
  - Reads an Excel file (`autoreconcilitation_training_terms_20251203_formatted.xlsx`) into a pandas DataFrame.

- 🧩 **Example Construction**
  - Uses `build_match_pairs(...)` to extract and format example pairs of:
    - `exactMatch`
    - `closeMatch`
    - `relatedMatch`  
  - These examples are concatenated into text blocks (`exact_text`, `close_text`, `related_text`) that describe each SKOS relation with real training examples.

- 🧠 **SKOSMatch Schema & LLM Setup**
  - Defines a `SKOSMatch` Pydantic model with fields:
    - `exact_match: bool`
    - `close_match: bool`
    - `related_match: bool`
    - `explanation: Optional[str]`
  - Each field’s description incorporates the corresponding example text, guiding the model’s behavior.
  - Configures a `ChatOpenAI` model (`gpt-5.1`, `temperature=0`) and wraps it with `.with_structured_output(SKOSMatch)` for structured JSON-like responses.

- 🔍 **Classification Function**
  - Implements `classify_skos_match(term_a, gen_def, term_b, onto_def)`:
    - Builds a prompt describing **Concept A** (term + generated definition) and **Concept B** (ontology term + definition).
    - Invokes the structured LLM to fill `SKOSMatch`.
    - Derives a `mapping_type`:
      - `"exact"` > `"close"` > `"related"` > `"none"` (in that priority order).
    - Returns a dictionary with:
      - `"mapping_type"`
      - `"explanation"`

- ✅ **Example Usage**
  - Demonstrates the classifier on:
    - Term A: *Campylobacter jejuni*
    - Term B: *campylobacteriosis*
  - Prints out the inferred SKOS mapping type and explanation.

# Installation and dependencies import

In [ ]:
!pip install langchain-openai langchain-core

In [8]:
from typing import Optional, Literal, Tuple
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent

from langchain_core.messages import SystemMessage, HumanMessage

from google.colab import userdata
import os

In [9]:
OPENAI_API_KEY=userdata.get('api-key-MicRisk')
os.environ["OPENAI_API_KEY"] =OPENAI_API_KEY

# Uploading examples for SKOS matching classes

In [10]:
import pandas as pd
from io import StringIO

df=pd.read_excel("/content/autoreconcilitation_training_terms_20251203_formatted.xlsx")

def build_match_pairs(
    df: pd.DataFrame,
    match_name: str,
    label_col: str,
    desc_col: str,
    term_col: str = "term",
    def_col: str = "definition",
):
    """
    Build plain-text pairs for a given SKOS match type.
    Returns a full text block as a string instead of printing.
    """
    buffer = StringIO()
    buffer.write(f"========== {match_name} pairs ==========\n\n")

    subset = df.dropna(subset=[label_col])

    for n, row in enumerate(subset.itertuples(index=False), start=1):
        term_a = getattr(row, term_col)
        def_a  = getattr(row, def_col)
        term_b = getattr(row, label_col)
        def_b  = getattr(row, desc_col)

        buffer.write(f"{n}) Term A: {term_a}\n")
        buffer.write(f"   Definition A: {def_a}\n")
        buffer.write(f"   Term B: {term_b}\n")
        buffer.write(f"   Definition B: {def_b}\n\n")

    return buffer.getvalue()

exact_text = build_match_pairs(
    df,
    match_name="exactMatch",
    label_col="exactMatch_label",
    desc_col="exactMatch_description"
)

close_text = build_match_pairs(
    df,
    match_name="closeMatch",
    label_col="closeMatch_label",
    desc_col="closeMatch_description"
)

related_text = build_match_pairs(
    df,
    match_name="relatedMatch",
    label_col="relatedMatch_label",
    desc_col="relatedMatch_description"
)

In [11]:
print(close_text)

========== closeMatch pairs ==========

1) Term A: Campylobacter jejuni
   Definition A: Gram-negative bacterial species that can cause diarrhoea in people
   Term B: Campylobacter 
   Definition B: genus of Gram-negative bacteria

2) Term A: Cow's milk
   Definition A: Whole, fresh, lacteal secretion obtained by the complete milking of one or more healthy cows
   Term B: milk
   Definition B: white liquid produced by the mammary glands of mammals

3) Term A: Highway
   Definition A: A main road, especially one connecting major towns or cities
   Term B: road
   Definition B: wide way leading from one place to another, especially one with a specially prepared surface which vehicles can use

4) Term A: Temperature
   Definition A: The specific degree of hotness or coldness of the body, regulated by the hypothalamus to stay within a specific range, usually around 37°C
   Term B: temperature
   Definition B: Temperature is the extent to which an object is hot.

5) Term A: Cattle
   Defini

# Define the structured output design and link it to the LLM

In [12]:
class SKOSMatch(BaseModel):
    """SKOS-style semantic relationship between two concepts."""

    exact_match: bool = Field(
        default=None,
        description=(
            f"""True if the two concepts can be used interchangeably across schemes.
They denote the same real-world concept, even if the wording differs.
This is symmetric and transitive.

EXACT MATCH EXAMPLES:
{exact_text}
"""
        ),
    )

    close_match: bool = Field(
        default=None,
        description=(
            f"""True if the two concepts are very similar and usually substitutable in most contexts,
but not strictly equivalent. Not transitive.

CLOSE MATCH EXAMPLES:
{close_text}
"""
        ),
    )

    related_match: bool = Field(
        default=None,
        description=(
            f"""True if the two concepts are associated but not synonymous.
Represents a non-hierarchical 'see also' relation.

RELATED MATCH EXAMPLES:
{related_text}
"""
        ),
    )

    explanation: Optional[str] = Field(
        default=None,
        description=(
            """Short explanation of why the chosen semantic relationship holds,
based on comparing terms and definitions."""
        ),
    )


# agent = create_agent(
#     model="gpt-5.1",
#     response_format=SKOSMatch,
# )

llm_skos = ChatOpenAI(
    model="gpt-5.1",
    temperature=0,
)
structured_llm_skos = llm_skos.with_structured_output(SKOSMatch)


def classify_skos_match(term_a: str, gen_def: str, term_b: str, onto_def: str):
    """
    Function to classify semantic relationships between two concepts into the
    SKOS concept: exact, close and related. The output is the matching type and
    the explanation
    """
    prompt=f"""
        You are comparing semantic similarities between two concepts. Each concept
        is represented by a term and its definition. Provide a concise summary
        explaining the semantic relationship between the concepts.

        Concept A:\n
          Term: {term_a}
          Definition (generated): {gen_def}
        Concept B:
          Term: {term_b}
          Definition (ontology): {onto_def}

      """

    messages = [
        HumanMessage(content=prompt),
    ]

    data: SKOSMatch = structured_llm_skos.invoke(messages)


     # Decide mapping_type with priority: exact > close > related
    if data.exact_match:
        mapping_type = "exact"
    elif data.close_match:
        mapping_type = "close"
    elif data.related_match:
        mapping_type = "related"
    else:
        mapping_type = "none"

    return {
        "mapping_type": mapping_type,
        "explanation": data.explanation or ""
    }


# Test the tool

In [13]:
termA= 'Campylobacter jejuni'
defA='Gram-negative bacterial species that can cause diarrhoea in people'

termB='campylobacteriosis'
defB='infections caused by a genus of Gram-negative bacteria'

output=classify_skos_match(termA, defA, termB, defB)
print(output)

{'mapping_type': 'related', 'explanation': 'Campylobacter jejuni is a specific Gram-negative bacterial species, whereas campylobacteriosis is the infection (disease) caused by Campylobacter bacteria, often including C. jejuni. They are causally and biologically related but are different types of entities (pathogen vs. disease), so the relation is associative rather than synonymous.'}
